# 05 - Netflix-paper recommend unseen movies

This notebook completes the Netflix-style RBM pipeline by generating recommendations.

Modeling assumption (important):
- A missing rating is treated as an unseen movie.
- Recommendations are generated only from missing entries.

## 1. Title and recommendation objective

This is the final Netflix-style RBM recommendation step.
The model infers user latent features from observed ratings and predicts ratings for missing entries only.
Missing ratings are treated as unseen movies in this setup.

## 2. Imports and data loading

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def resolve_project_root() -> Path:
    root = Path.cwd().resolve()
    if root.name == "notebooks":
        root = root.parent
    return root


def load_ratings_table() -> tuple[pd.DataFrame, Path]:
    root = resolve_project_root()
    candidates = [
        root / "data" / "processed" / "train_ratings.csv",
        root / "data" / "processed" / "ratings.csv",
        root / "data" / "processed" / "rating.csv",
        root / "data" / "rating.csv",
        root / "data" / "ratings.csv",
    ]

    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    data_dir = root / "data"
    search_dirs = [data_dir / "processed", data_dir]
    for directory in search_dirs:
        if not directory.exists():
            continue
        for path in sorted(directory.glob("*.csv")):
            try:
                df = pd.read_csv(path)
            except Exception:
                continue
            if {"userId", "movieId", "rating"}.issubset(df.columns):
                return df[["userId", "movieId", "rating"]].copy(), path

    raise FileNotFoundError("No ratings table found with columns userId, movieId, rating.")


def load_movie_metadata() -> pd.DataFrame | None:
    root = resolve_project_root()
    candidates = [
        root / "data" / "movie.csv",
        root / "data" / "movies.csv",
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path)
            if {"movieId", "title"}.issubset(df.columns):
                return df[["movieId", "title"]].copy()
    return None


ratings_df, ratings_path = load_ratings_table()
movies_df = load_movie_metadata()

print(f"Using ratings table: {ratings_path}")
print("Ratings shape:", ratings_df.shape)
if movies_df is not None:
    print("Movies metadata shape:", movies_df.shape)
    display(movies_df.head())
else:
    print("Movies metadata not found.")

ratings_df.head()

Using ratings table: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/train_ratings.csv
Ratings shape: (16000210, 3)
Movies metadata shape: (27278, 2)


,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)


,userId,movieId,rating
0,28507,1176,4.0
1,131160,1079,3.0
2,131160,47,5.0
3,131160,21,3.0
4,85252,45,3.0


## 3. Rebuild the Netflix-style RBM setup

We rebuild the same user and movie subsets and run a short CD-1 training block to get a consistent trained state.
This keeps the notebook self-contained and runnable end-to-end.

In [2]:
rating_levels = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
K = len(rating_levels)
rating_to_index = {float(r): idx for idx, r in enumerate(rating_levels)}

# Subset sizes for a runnable demo
max_users = 1000
max_movies = 1000
min_ratings_per_user = 20

user_counts = ratings_df.groupby("userId").size().reset_index(name="count")
eligible_users = user_counts[user_counts["count"] >= min_ratings_per_user]
eligible_users = eligible_users.sort_values(["count", "userId"], ascending=[False, True])
selected_user_ids = eligible_users["userId"].head(max_users).tolist()

subset_df = ratings_df[ratings_df["userId"].isin(selected_user_ids)].copy()
movie_counts = subset_df.groupby("movieId").size().reset_index(name="count")
selected_movie_ids = movie_counts.sort_values("count", ascending=False)["movieId"].head(max_movies).tolist()

subset_df = subset_df[subset_df["movieId"].isin(selected_movie_ids)].copy()

user_id_to_index = {uid: idx for idx, uid in enumerate(sorted(subset_df["userId"].unique()))}
movie_id_to_index = {mid: idx for idx, mid in enumerate(sorted(selected_movie_ids))}
index_to_movie_id = {idx: mid for mid, idx in movie_id_to_index.items()}

num_users = len(user_id_to_index)
num_movies = len(movie_id_to_index)

V = np.zeros((num_users, K, num_movies), dtype=np.float32)
mask = np.zeros((num_users, num_movies), dtype=np.float32)

for row in subset_df.itertuples(index=False):
    uid = row.userId
    mid = row.movieId
    rating = float(row.rating)
    if rating not in rating_to_index:
        continue
    u_idx = user_id_to_index[uid]
    m_idx = movie_id_to_index[mid]
    k_idx = rating_to_index[rating]
    V[u_idx, k_idx, m_idx] = 1.0
    mask[u_idx, m_idx] = 1.0

print("Users in subset:", num_users)
print("Movies in subset:", num_movies)
print("V shape:", V.shape)
print("Mask shape:", mask.shape)
print("Observed ratings count:", int(mask.sum()))

Users in subset: 1000
Movies in subset: 1000
V shape: (1000, 10, 1000)
Mask shape: (1000, 1000)
Observed ratings count: 630479


Hidden activation (Eq. 6, Netflix RBM paper):

$$
p(h_j = 1 \mid v) = \sigma\left(b_j + \sum_{i,k} v_i^k W_{ijk}\right)
$$

This computes the probability of latent feature activation given visible ratings.

Visible reconstruction (Eq. 7, Netflix RBM paper):

$$
p(v_i^k = 1 \mid h) = \frac{\exp\left(b_i^k + \sum_j h_j W_{ijk}\right)}{\sum_l \exp\left(b_i^l + \sum_j h_j W_{ijl}\right)}
$$

This gives the probability of each rating level for movie \(i\).

In [3]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def softmax_over_k(x: np.ndarray) -> np.ndarray:
    x_max = x.max(axis=1, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / exp_x.sum(axis=1, keepdims=True)


def hidden_probs_from_visible(V_batch: np.ndarray, W: np.ndarray, b_h: np.ndarray) -> np.ndarray:
    activation = np.einsum("bkm,kmf->bf", V_batch, W) + b_h
    return sigmoid(activation)


def visible_probs_from_hidden(h_batch: np.ndarray, W: np.ndarray, b_v: np.ndarray) -> np.ndarray:
    scores = np.einsum("bf,kmf->bkm", h_batch, W) + b_v[None, :, :]
    return softmax_over_k(scores)


def cd1_update(V_batch: np.ndarray, mask_batch: np.ndarray, W: np.ndarray, b_v: np.ndarray, b_h: np.ndarray, rng: np.random.Generator):
    h_probs_pos = hidden_probs_from_visible(V_batch, W, b_h)
    h_states = rng.binomial(1, h_probs_pos).astype(np.float32)

    v_probs_neg_full = visible_probs_from_hidden(h_states, W, b_v)
    v_probs_neg = v_probs_neg_full * mask_batch[:, None, :]
    h_probs_neg = hidden_probs_from_visible(v_probs_neg, W, b_h)

    V_pos = V_batch * mask_batch[:, None, :]
    V_neg = v_probs_neg

    pos_stats = np.einsum("bkm,bf->kmf", V_pos, h_probs_pos)
    neg_stats = np.einsum("bkm,bf->kmf", V_neg, h_probs_neg)

    grad_W = pos_stats - neg_stats
    grad_b_v = (V_pos - V_neg).sum(axis=0)
    grad_b_h = (h_probs_pos - h_probs_neg).sum(axis=0)

    return grad_W, grad_b_v, grad_b_h

In [4]:
F = 50
rng = np.random.default_rng(42)

W = rng.normal(loc=0.0, scale=0.01, size=(K, num_movies, F)).astype(np.float32)
b_v = np.zeros((K, num_movies), dtype=np.float32)
b_h = np.full((F,), -1.0, dtype=np.float32)

learning_rate = 0.005
weight_decay = 1e-4
epochs = 5
batch_size = 50

num_batches = int(np.ceil(num_users / batch_size))

for epoch in range(1, epochs + 1):
    perm = rng.permutation(num_users)
    epoch_loss = 0.0
    total_observed = 0.0

    for b in range(num_batches):
        batch_idx = perm[b * batch_size : (b + 1) * batch_size]
        V_batch = V[batch_idx]
        mask_batch = mask[batch_idx]

        grad_W, grad_b_v, grad_b_h = cd1_update(V_batch, mask_batch, W, b_v, b_h, rng)
        grad_W = grad_W - weight_decay * W

        W += learning_rate * grad_W / batch_idx.size
        b_v += learning_rate * grad_b_v / batch_idx.size
        b_h += learning_rate * grad_b_h / batch_idx.size

        # Quick masked reconstruction loss for monitoring
        v_probs_neg = visible_probs_from_hidden(
            rng.binomial(1, hidden_probs_from_visible(V_batch, W, b_h)).astype(np.float32),
            W,
            b_v,
        )
        v_probs_neg = v_probs_neg * mask_batch[:, None, :]
        eps = 1e-9
        loss = -(V_batch * np.log(np.clip(v_probs_neg, eps, 1.0))).sum(axis=1)
        loss = (loss * mask_batch).sum()
        epoch_loss += loss
        total_observed += mask_batch.sum()

    avg_loss = epoch_loss / max(total_observed, 1.0)
    print(f"Epoch {epoch:02d} | Recon loss: {avg_loss:.4f}")

Epoch 01 | Recon loss: 2.2490
Epoch 02 | Recon loss: 2.0527
Epoch 03 | Recon loss: 1.9470
Epoch 04 | Recon loss: 1.9015
Epoch 05 | Recon loss: 1.8776


## 4. Predict rating distributions for missing entries

Expected rating (Eq. 10, Netflix RBM paper):

$$
\hat r_i = \sum_{k=1}^{K} r_k \; p(v_i^k = 1 \mid h)
$$

The predicted rating is the expectation over rating probabilities.

In [5]:
def expected_ratings_from_probs(P_visible: np.ndarray, rating_levels: list[float]) -> np.ndarray:
    rating_array = np.array(rating_levels, dtype=np.float32)
    return (rating_array[:, None] * P_visible).sum(axis=0)


def get_user_recommendations(user_id: int, top_k: int = 10) -> pd.DataFrame:
    u_idx = user_id_to_index[user_id]
    V_u = V[u_idx]
    mask_u = mask[u_idx]

    observed_idx = np.where(mask_u > 0)[0]
    missing_idx = np.where(mask_u == 0)[0]

    h_probs_u = hidden_probs_from_visible(V_u[None, :, :], W, b_h)[0]
    P_u = visible_probs_from_hidden(h_probs_u[None, :], W, b_v)[0]

    expected_u = expected_ratings_from_probs(P_u, rating_levels)

    # Rank only missing (unseen) movies
    missing_scores = expected_u[missing_idx]
    top_indices = missing_idx[np.argsort(missing_scores)[::-1][:top_k]]

    recs = pd.DataFrame(
        {
            "movieId": [index_to_movie_id[i] for i in top_indices],
            "predicted_rating": expected_u[top_indices],
            "top_rating_level_probability": P_u[:, top_indices].max(axis=0),
        }
    )

    if movies_df is not None:
        recs = recs.merge(movies_df, on="movieId", how="left")
        recs = recs[["movieId", "title", "predicted_rating", "top_rating_level_probability"]]

    summary = {
        "observed_movies": int(len(observed_idx)),
        "missing_movies": int(len(missing_idx)),
    }

    return recs, summary, set(index_to_movie_id[i] for i in observed_idx)


example_users = list(user_id_to_index.keys())[:2]

for uid in example_users:
    recs, summary, observed_movie_ids = get_user_recommendations(uid, top_k=10)
    print(f"\nUser {uid}")
    print("Observed movies:", summary["observed_movies"])
    print("Unseen candidate movies:", summary["missing_movies"])
    display(recs)

    # Sanity check: ensure no observed movie appears in recommendations
    overlap = set(recs["movieId"]) & observed_movie_ids
    print("Observed overlap in recommendations:", len(overlap))


User 116
Observed movies: 570
Unseen candidate movies: 430


,movieId,title,predicted_rating,top_rating_level_probability
0,912,Casablanca (1942),4.044138,0.430018
1,1221,"Godfather: Part II, The (1974)",4.042977,0.447360
2,1193,One Flew Over the Cuckoo's Nest (1975),3.956747,0.321211
3,904,Rear Window (1954),3.940989,0.343818
4,923,Citizen Kane (1941),3.930261,0.404915
5,4226,Memento (2000),3.927336,0.313138
6,1252,Chinatown (1974),3.919259,0.370172
7,1219,Psycho (1960),3.913159,0.344014
8,908,North by Northwest (1959),3.896144,0.322944
9,1207,To Kill a Mockingbird (1962),3.864836,0.329156


Observed overlap in recommendations: 0

User 156
Observed movies: 671
Unseen candidate movies: 329


,movieId,title,predicted_rating,top_rating_level_probability
0,260,Star Wars: Episode IV - A New Hope (1977),4.022852,0.401973
1,1213,Goodfellas (1990),3.979660,0.346873
2,1252,Chinatown (1974),3.972783,0.387266
3,1219,Psycho (1960),3.962869,0.359023
4,908,North by Northwest (1959),3.945391,0.335059
5,2019,Seven Samurai (Shichinin no samurai) (1954),3.936178,0.408643
6,7153,"Lord of the Rings: The Return of the King, The...",3.931551,0.329866
7,1197,"Princess Bride, The (1987)",3.832129,0.303914
8,1203,12 Angry Men (1957),3.826629,0.303194
9,4973,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",3.823721,0.283786


Observed overlap in recommendations: 0


## 5. Evaluate prediction quality on observed ratings

We evaluate RMSE and MAE only on observed ratings using the mask.

In [6]:
rating_levels_array = np.array(rating_levels, dtype=np.float32)

# True ratings from one-hot visible tensor
true_ratings = (rating_levels_array[None, :, None] * V).sum(axis=1)

# Predicted ratings from reconstructed probabilities
pred_ratings = np.zeros_like(true_ratings)
for u_idx in range(num_users):
    h_probs_u = hidden_probs_from_visible(V[u_idx : u_idx + 1], W, b_h)[0]
    P_u = visible_probs_from_hidden(h_probs_u[None, :], W, b_v)[0]
    pred_ratings[u_idx] = expected_ratings_from_probs(P_u, rating_levels)

# Evaluate only observed entries
observed_mask = mask == 1
y_true = true_ratings[observed_mask]
y_pred = pred_ratings[observed_mask]

rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
mae = float(np.mean(np.abs(y_pred - y_true)))

print("Evaluation on observed ratings")
print("Number of ratings:", len(y_true))
print("RMSE:", rmse)
print("MAE:", mae)

Evaluation on observed ratings
Number of ratings: 630479
RMSE: 0.9243324995040894
MAE: 0.7421854138374329


## 6. Baseline evaluation on observed ratings

Baselines are necessary to judge whether the RBM improves over simple heuristics.
A recommender model is only meaningful if it outperforms average-based predictors.

In [7]:
from IPython.display import Markdown, display

# Baseline predictions on the same observed entries
observed_mask = mask == 1

# Global mean baseline
global_mean = float(y_true.mean())

# User mean baseline components
user_sum = (true_ratings * mask).sum(axis=1)
user_count = mask.sum(axis=1)
user_mean = user_sum / np.maximum(user_count, 1.0)

# Movie mean baseline components
movie_sum = (true_ratings * mask).sum(axis=0)
movie_count = mask.sum(axis=0)
movie_mean = movie_sum / np.maximum(movie_count, 1.0)
movie_mean = np.where(movie_count > 0, movie_mean, global_mean)

# Baseline predictions for observed entries
pred_global = np.full_like(true_ratings, global_mean)
pred_movie = movie_mean[None, :]
pred_movie_full = np.broadcast_to(pred_movie, true_ratings.shape)

pred_user_movie = user_mean[:, None] + movie_mean[None, :] - global_mean
pred_user_movie = np.clip(pred_user_movie, 0.5, 5.0)

# Metrics helper
def rmse_mae(y_t: np.ndarray, y_p: np.ndarray) -> tuple[float, float]:
    rmse_val = float(np.sqrt(np.mean((y_p - y_t) ** 2)))
    mae_val = float(np.mean(np.abs(y_p - y_t)))
    return rmse_val, mae_val

rbm_rmse, rbm_mae = rmse, mae

global_rmse, global_mae = rmse_mae(y_true, pred_global[observed_mask])
movie_rmse, movie_mae = rmse_mae(y_true, pred_movie_full[observed_mask])
user_movie_rmse, user_movie_mae = rmse_mae(y_true, pred_user_movie[observed_mask])

summary_df = pd.DataFrame(
    {
        "model": ["RBM", "Global mean", "Movie mean", "User+Movie mean"],
        "RMSE": [rbm_rmse, global_rmse, movie_rmse, user_movie_rmse],
        "MAE": [rbm_mae, global_mae, movie_mae, user_movie_mae],
    }
)

display(summary_df)

# Interpretation in markdown based on metrics
best_baseline_rmse = min(global_rmse, movie_rmse, user_movie_rmse)
if rbm_rmse < best_baseline_rmse:
    verdict = "RBM beats the baselines on RMSE."
elif np.isclose(rbm_rmse, best_baseline_rmse, rtol=1e-3):
    verdict = "RBM is only slightly better or comparable to the strongest baseline."
else:
    verdict = "RBM does not beat the strongest baseline on RMSE."

interpretation = (
    f"**Interpretation:** {verdict} "
    "Use the table above to compare RMSE/MAE against simple average-based predictors."
)

display(Markdown(interpretation))

,model,RMSE,MAE
0,RBM,0.924332,0.742185
1,Global mean,1.023983,0.826855
2,Movie mean,0.903847,0.709750
3,User+Movie mean,0.820152,0.631851


**Interpretation:** RBM does not beat the strongest baseline on RMSE. Use the table above to compare RMSE/MAE against simple average-based predictors.

## 7. Recommend unseen movies

Recommendations are ranked only from missing entries (unseen movies in this setup).
Top-10 recommendations are shown per user.

## 8. Wrap-up

Previous notebooks built and trained the Netflix-style RBM.
This notebook completes the recommendation pipeline.
The model uses observed ratings to infer latent features and predicts ratings for missing entries treated as unseen movies.
The top-ranked missing movies become the final recommendations.